# Fine-tune a tiny receipt reader (AgentLab Step 4.3)

This notebook teaches **Gemma 3 270M** one job: read a receipt's text and answer with its fields as JSON (vendor, date, subtotal, GST, other tax, tip, total).

- **Data:** 2,000 **fictional** receipts made by [`make_training_data.py`](https://github.com/GurminderKaur/AgentLab/blob/main/07-fine-tuning/data/make_training_data.py), plus 200 held out for checking. Nothing real is uploaded.
- **Method:** LoRA with [Unsloth](https://github.com/unslothai/unsloth): only a small add-on is trained; the original model stays frozen.
- **Output:** one `receipt-reader-Q8_0.gguf` file (~0.3 GB) to run in Ollama on my laptop.

**Before you start:** *Runtime → Change runtime type → **T4 GPU** → Save.* Then run the cells top to bottom (▶ on each, or *Runtime → Run all*). It takes about 15–25 minutes in total.

## 1. Install the tools (about 2–3 minutes)

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

## 2. Get the fictional training data from GitHub

In [2]:
!wget -q -O train.jsonl https://raw.githubusercontent.com/GurminderKaur/AgentLab/main/07-fine-tuning/data/train.jsonl
!wget -q -O heldout.jsonl https://raw.githubusercontent.com/GurminderKaur/AgentLab/main/07-fine-tuning/data/heldout.jsonl
import json
heldout = [json.loads(l) for l in open("heldout.jsonl")]
print(sum(1 for _ in open("train.jsonl")), "training receipts,", len(heldout), "held out")
print(heldout[0]["messages"][0]["content"][-400:])
print("→", heldout[0]["messages"][1]["content"])

2000 training receipts, 200 held out
ed. "gst" is only the amount on a GST or HST line. Use 0 for anything that isn't printed.

Receipt:
SIGNAL PRINT & COPY
599 Fictional Ave NW
Regina, SK
2026/10/08
Poster 24x36 41.60
Business cards x200 96.13
Banner 3x6 ft 132.82
Flyers x250 121.57
Subtotal 392.12
G.S.T. 19.61
PST 23.53
TOTAL $435.26
VISA ****6031 APPROVED
Thanks for shopping!
*** SAMPLE RECEIPT - FOR TESTING ***
fictional business
→ {"vendor": "SIGNAL PRINT & COPY", "date": "2026-10-08", "subtotal": 392.12, "gst": 19.61, "other_tax": 23.53, "tip": 0.0, "total": 435.26}


## 3. Load Gemma 3 270M and add a LoRA adapter

Only the adapter (a small set of new numbers) will be trained.

In [3]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-270m-it",
    max_seq_length = 2048,
    load_in_4bit = False,
    load_in_8bit = False,
    full_finetuning = False,
)
model = FastModel.get_peft_model(
    model,
    r = 128,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template = "gemma3")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1128 17:33:35.936000 471 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1128 17:33:36.005000 471 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.12: Fast Gemma3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


## 4. A scorer: how many fields does the model get right?

Same rules as ReceiptBench on my laptop: amounts to the cent, date exact, vendor by name.

In [4]:
import re
FIELDS = ["vendor", "date", "subtotal", "gst", "other_tax", "tip", "total"]

def ask(user_text):
    text = tokenizer.apply_chat_template([{"role": "user", "content": user_text}], tokenize = False, add_generation_prompt = True).removeprefix('<bos>')
    inputs = tokenizer(text, return_tensors = "pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens = 160, do_sample = False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True)

def norm(s): return re.sub(r"[^a-z0-9]", "", str(s).lower())
def num(v):
    try: return float(re.sub(r"[^0-9.\-]", "", str(v)) or 0)
    except ValueError: return 0.0

def score(examples, show = 2):
    hits = {f: 0 for f in FIELDS}
    for i, ex in enumerate(examples):
        want = json.loads(ex["messages"][1]["content"])
        reply = ask(ex["messages"][0]["content"])
        m = re.search(r"\{.*\}", reply, re.S)
        try: got = json.loads(m.group(0)) if m else {}
        except json.JSONDecodeError: got = {}
        ok = {
            "vendor": len(norm(got.get("vendor", ""))) > 4 and (norm(got.get("vendor", "")) in norm(want["vendor"]) or norm(want["vendor"]) in norm(got.get("vendor", ""))),
            "date": got.get("date") == want["date"],
            **{f: abs(num(got.get(f, 0)) - want[f]) < 0.005 for f in ["subtotal", "gst", "other_tax", "tip", "total"]},
        }
        for f in FIELDS: hits[f] += ok[f]
        if i < show: print("model said:", reply.strip()[:200], "\nright was:", ex["messages"][1]["content"], "\n")
    n = len(examples)
    print(", ".join(f"{f} {hits[f]}/{n}" for f in FIELDS), f"· all fields right on {sum(hits.values())}/{n * len(FIELDS)}")
    return hits

## 5. Before training: score 50 held-out receipts

Expect poor results: in ReceiptBench the untrained model copied the example shape (`"..."`, `"YYYY-MM-DD"`, zeros).

In [5]:
before = score(heldout[:50])

model said: ```json
{"vendor": "...", "date": "YYYY-MM-DD", "subtotal": 0.00, "gst": 0.00, "other_tax": 0.00, "tip": 0.00, "total": 0.00}
``` 
right was: {"vendor": "SIGNAL PRINT & COPY", "date": "2026-10-08", "subtotal": 392.12, "gst": 19.61, "other_tax": 23.53, "tip": 0.0, "total": 435.26} 

model said: ```json
{"vendor": "...", "date": "YYYY-MM-DD", "subtotal": 0.00, "gst": 0.00, "other_tax": 0.00, "tip": 0.00, "total": 0.00}
``` 
right was: {"vendor": "Aspen Grind Coffee", "date": "2026-03-01", "subtotal": 5.67, "gst": 0.28, "other_tax": 0.0, "tip": 0.0, "total": 5.95} 

vendor 3/50, date 2/50, subtotal 0/50, gst 8/50, other_tax 35/50, tip 48/50, total 0/50 · all fields right on 96/350


## 6. Train (about 5–10 minutes on a T4)

The loss (how wrong the model's answers are) should fall quickly and then level off. Only the **answers** are scored during training, not the questions (`train_on_responses_only`).

In [6]:
from datasets import load_dataset
dataset = load_dataset("json", data_files = "train.jsonl", split = "train")
def formatting(examples):
    return {"text": [tokenizer.apply_chat_template(m, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for m in examples["messages"]]}
dataset = dataset.map(formatting, batched = True)

from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 1,
        warmup_steps = 10,
        num_train_epochs = 2,          # two passes over the 2,000 receipts
        learning_rate = 1e-4,
        logging_steps = 25,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer)
stats = trainer.train()
print(f"Trained in {stats.metrics['train_runtime'] / 60:.1f} minutes")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2000 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/2000 [00:00<?, ? examples/s]

... (2 outputs left out)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 340 | Num Epochs = 2 | Total steps = 680
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 30,375,936 of 298,474,112 (10.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
25,0.084400
50,0.003900
75,0.001400
100,0.002900
125,0.000700
150,0.000000
175,0.000000
200,0.000000
225,0.000000
250,0.000000


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Trained in 10.5 minutes


## 7. After training: score all 200 held-out receipts

These were never trained on. A big jump here means the model learned the rules, not just the examples.

In [7]:
after = score(heldout, show = 3)

model said: {"vendor": "SIGNAL PRINT & COPY", "date": "2026-10-08", "subtotal": 392.12, "gst": 19.61, "other_tax": 23.53, "tip": 0.0, "total": 435.26} 
right was: {"vendor": "SIGNAL PRINT & COPY", "date": "2026-10-08", "subtotal": 392.12, "gst": 19.61, "other_tax": 23.53, "tip": 0.0, "total": 435.26} 

model said: {"vendor": "Aspen Grind Coffee", "date": "2026-03-01", "subtotal": 5.67, "gst": 0.28, "other_tax": 0.0, "tip": 0.0, "total": 5.95} 
right was: {"vendor": "Aspen Grind Coffee", "date": "2026-03-01", "subtotal": 5.67, "gst": 0.28, "other_tax": 0.0, "tip": 0.0, "total": 5.95} 

model said: {"vendor": "Sunrise Fresh Market", "date": "2026-01-03", "subtotal": 44.63, "gst": 0.0, "other_tax": 0.0, "tip": 0.0, "total": 44.63} 
right was: {"vendor": "Sunrise Fresh Market", "date": "2026-01-03", "subtotal": 44.63, "gst": 0.0, "other_tax": 0.0, "tip": 0.0, "total": 44.63} 

vendor 200/200, date 200/200, subtotal 200/200, gst 200/200, other_tax 200/200, tip 200/200, total 200/200 · all f

## 8. Export to one GGUF file for Ollama, and download it (~0.3 GB)

Your browser will ask where to save `receipt-reader-Q8_0.gguf`. Save it into your `AgentLab\\07-fine-tuning` folder, next to the `Modelfile`.

In [8]:
import glob, shutil
model.save_pretrained_gguf("receipt_reader", tokenizer, quantization_method = "Q8_0")
ggufs = sorted(glob.glob("receipt_reader*/*.gguf") + glob.glob("receipt_reader*.gguf") + glob.glob("*.gguf"), key = os.path.getsize, reverse = True)
print("GGUF files found:", ggufs)
shutil.copy(ggufs[0], "receipt-reader-Q8_0.gguf")
print(f"receipt-reader-Q8_0.gguf: {os.path.getsize('receipt-reader-Q8_0.gguf') / 1e6:.0f} MB")
from google.colab import files
files.download("receipt-reader-Q8_0.gguf")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `receipt_reader`: 100%|██████████| 1/1 [00:00<00:00, 3721.65it/s]


Successfully copied all 1 files from cache to `receipt_reader`
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `receipt_reader`: 100%|██████████| 1/1 [00:00<00:00, 164.25it/s]


Successfully copied all 1 files from cache to `receipt_reader`


... (4 outputs left out)

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: Initial conversion completed! Files: ['receipt_reader_gguf/gemma-3-270m-it.Q8_0.gguf']
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['receipt_reader_gguf/gemma-3-270m-it.Q8_0.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model receipt_reader_gguf/gemma-3-270m-it.Q8_0.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to receipt_reader_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f receipt_reader_gguf/Modelfile
GGUF files found: ['receipt_reader_gguf/gemma-3-270m-it.Q8_0.gguf']
receipt-reader-Q8_0.gguf: 292 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Next, on the laptop (Step 4.4)

In PowerShell, in your AgentLab folder:

```powershell
git pull
cd 07-fine-tuning
ollama create receipt-reader -f Modelfile
cd ReceiptBench
dotnet run -- extract receipt-reader
```

Then compare with the "before" numbers (11/42) and Gemma 4 (42/42).